In [11]:
import sqlite3
from langgraph.graph import StateGraph, START, MessagesState,END 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from langgraph.checkpoint.sqlite import SqliteSaver

In [2]:
my_secret_key = os.getenv("GROQ_API_KEY")
load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-20b",  # Current standard for high-accuracy reasoning workflows
    api_key=my_secret_key,
    temperature=0.3
)

In [3]:
def call_mode(state:MessagesState):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

In [4]:
graph = StateGraph(MessagesState)

#add node 
graph.add_node('call_mode',call_mode)

#add edges 
graph.add_edge(START,'call_mode')
graph.add_edge('call_mode',END)

In [12]:
# 1. Open a SQLite connection (check_same_thread=False is required for SQLite with threads)
conn = sqlite3.connect("chat_history.db", check_same_thread=False)

# 2. Create the SqliteSaver checkpointer instance
checkpointer = SqliteSaver(conn)

In [13]:
workflow = graph.compile(checkpointer=checkpointer)

In [14]:
# Define the thread configuration
config = {"configurable": {"thread_id": "user_session_123"}}

# Invoke the graph
response = workflow.invoke(
    {"messages": [{"role": "user", "content": "Hi, My name is Sri Rama"}]},
    config
)

print(response)

{'messages': [HumanMessage(content='Hi, My name is Sri Rama', additional_kwargs={}, response_metadata={}, id='ab54f150-7ed5-4dd5-bf11-1e96ba092e0f'), AIMessage(content='Hello Sri\u202fRama! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond politely, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 78, 'total_tokens': 114, 'completion_time': 0.036372264, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.004233979, 'prompt_tokens_details': None, 'queue_time': 0.31008759, 'total_time': 0.040606243}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1ed00-ee6a-701e-8f7c-cc5ccf671f02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 36, 'total_tokens': 114, 'output_token_details': {'reasoning': 12}}

In [15]:
# Later on, retrieve the active state for that thread
current_state = workflow.get_state(config)
print("Previous messages:", current_state.values.get("messages"))

Previous messages: [HumanMessage(content='Hi, My name is Sri Rama', additional_kwargs={}, response_metadata={}, id='ab54f150-7ed5-4dd5-bf11-1e96ba092e0f'), AIMessage(content='Hello Sri\u202fRama! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond politely, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 78, 'total_tokens': 114, 'completion_time': 0.036372264, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.004233979, 'prompt_tokens_details': None, 'queue_time': 0.31008759, 'total_time': 0.040606243}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1ed00-ee6a-701e-8f7c-cc5ccf671f02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 36, 'total_tokens': 114, 'output_token_details': {'reasoning'

In [18]:
# Use the SAME thread_id as before
config = {"configurable": {"thread_id": "user_session_123"}}

# Send the next message
response = workflow.invoke(
    {"messages": [{"role": "user", "content": "What was the first thing I said to you?"}]},
    config
)

# Print only the latest AI reply
print(response["messages"][-1].content)

The first thing you said to me was:  
**“Hi, My name is Sri Rama.”**
